In [1]:
import geopandas as gpd
import os
import requests
from shapely.validation import make_valid

# Cargar el límite oficial de Usaquén
usaquen_oficial = gpd.read_file("../data/raw/limites/loca.json")

usaquen_filtrado = usaquen_oficial[
    usaquen_oficial["LocNombre"] == "USAQUEN"
]

usaquen_filtrado = usaquen_filtrado.to_crs(epsg=4326)

usaquen_polygon = usaquen_filtrado.geometry.iloc[0]

if not usaquen_polygon.is_valid:
    usaquen_polygon = make_valid(usaquen_polygon)

print(f"Polígono válido: {usaquen_polygon.is_valid}")
print(usaquen_polygon.geom_type)

Polígono válido: True
Polygon


In [2]:
coords = list(usaquen_polygon.exterior.coords)

poly_coords = " ".join(
    f"{lat} {lon}"
    for lon, lat in coords
)

print(poly_coords[:500])

4.6645853120000424 -74.01116193799993 4.664600325000038 -74.01116635999995 4.664670392000062 -74.0111970449999 4.6647556270000905 -74.0112404919999 4.664802361000056 -74.01126767399995 4.664843971000039 -74.01128191799995 4.664872431000049 -74.01129356699994 4.664905297000075 -74.01130481899992 4.664934909000067 -74.01131778699994 4.664984942000046 -74.01134688799993 4.665016592000086 -74.0113534969999 4.665055760000087 -74.01137346099995 4.665099846000089 -74.01138855699992 4.665135217000056 -7


In [3]:
ruta_archivo = '../data/processed/usaquen.osm.xml'

query = f"""
[out:xml][timeout:180];
(
  way["highway"](poly:"{poly_coords}");
);
(._;>;);
out meta;
"""

response = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={'data': query.encode('utf-8')},
    headers={"User-Agent": "MiAplicacionGeografica/1.0"},
    timeout=300
)

if response.status_code == 200:
    #Guarda el contenido en disco
    with open(ruta_archivo, 'wb') as f:
        f.write(response.content)
    
    print(f"Archivo guardado en: {os.path.abspath(ruta_archivo)}\n")

    #primeras líneas
    print("Contenido del archivo")
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        for i in range(10):
            print(f.readline(), end='')
else:
    print(f"Error HTTP {response.status_code}: {response.text[:200]}")

Archivo guardado en: c:\Users\USUARIO\Downloads\simulacion-movilidad-urbana\data\processed\usaquen.osm.xml

Contenido del archivo
<?xml version="1.0" encoding="UTF-8"?>
<osm version="0.6" generator="Overpass API 0.7.62.11 87bfad18">
<note>The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.</note>
<meta osm_base="2026-08-21T18:47:06Z"/>

  <node id="253845462" lat="4.6752483" lon="-74.0243597" version="18" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253845500" lat="4.6752084" lon="-74.0244588" version="12" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253845875" lat="4.6757979" lon="-74.0264972" version="18" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253846345" lat="4.6680374" lon="-74.0124907" version="9" timestamp="2020-06-01T15:55:02Z" changeset="86053783" uid="

In [6]:
import osmnx as ox
import pandas as pd

G = ox.graph_from_xml(
    "../data/processed/usaquen.osm.xml",
    simplify=False
)

nodes, edges = ox.graph_to_gdfs(G)

print("Total de aristas:", len(edges))

print("\n--- MAXSPEED ---")
print(edges["maxspeed"].value_counts(dropna=False).head(30))

print("\n--- LANES ---")
print(edges["lanes"].value_counts(dropna=False).head(30))

print("\n--- FALTANTES ---")

total = len(edges)

print(
    f"Sin maxspeed: {edges['maxspeed'].isna().sum()} "
    f"({edges['maxspeed'].isna().mean()*100:.2f}%)"
)

print(
    f"Sin lanes: {edges['lanes'].isna().sum()} "
    f"({edges['lanes'].isna().mean()*100:.2f}%)"
)

Total de aristas: 98563

--- MAXSPEED ---
maxspeed
NaN    81417
30     15160
20       476
40       443
10       410
50       389
60       268
Name: count, dtype: int64

--- LANES ---
lanes
NaN    76103
2      17408
1       2755
3       1984
4        259
5         35
6         15
20         4
Name: count, dtype: int64

--- FALTANTES ---
Sin maxspeed: 81417 (82.60%)
Sin lanes: 76103 (77.21%)


In [7]:
import xml.etree.ElementTree as ET
import pandas as pd
from collections import Counter

NET_FILE = "../data/processed/usaquen.net.xml"

tree = ET.parse(NET_FILE)
root = tree.getroot()

edges = []
lanes = []

for edge in root.findall("edge"):
    # Ignorar edges internos de intersecciones
    if edge.get("function") == "internal":
        continue

    edge_id = edge.get("id")
    edge_type = edge.get("type")
    edge_from = edge.get("from")
    edge_to = edge.get("to")

    edge_lanes = edge.findall("lane")

    for lane in edge_lanes:
        lanes.append({
            "edge_id": edge_id,
            "type": edge_type,
            "from": edge_from,
            "to": edge_to,
            "lane_id": lane.get("id"),
            "index": lane.get("index"),
            "speed": lane.get("speed"),
            "length": lane.get("length"),
            "allow": lane.get("allow"),
            "disallow": lane.get("disallow"),
            "width": lane.get("width")
        })

    edges.append({
        "edge_id": edge_id,
        "type": edge_type,
        "from": edge_from,
        "to": edge_to,
        "num_lanes": len(edge_lanes)
    })

edges_df = pd.DataFrame(edges)
lanes_df = pd.DataFrame(lanes)

print(f"Edges externos: {len(edges_df):,}")
print(f"Carriles: {len(lanes_df):,}")

Edges externos: 28,070
Carriles: 32,012


In [8]:
print("\n--- VELOCIDADES ---")

print(
    lanes_df["speed"]
    .astype(float)
    .describe()
)


--- VELOCIDADES ---
count    32012.000000
mean         9.040711
std          7.032922
min          1.390000
25%          2.780000
50%          8.330000
75%         13.890000
max         27.780000
Name: speed, dtype: float64


In [9]:
print(
    "\nVelocidades más frecuentes:"
)

print(
    lanes_df["speed"]
    .astype(float)
    .value_counts()
    .sort_index()
)


Velocidades más frecuentes:
speed
1.39      237
2.78     7975
5.56     7405
8.33     7854
11.11     336
13.89    4692
16.67     244
22.22     624
27.78    2645
Name: count, dtype: int64


In [10]:
velocidades = lanes_df["speed"].astype(float)

print(
    pd.DataFrame({
        "speed_m_s": velocidades.value_counts().sort_index().index,
        "speed_km_h": velocidades.value_counts().sort_index().index * 3.6,
        "cantidad": velocidades.value_counts().sort_index().values
    })
)

   speed_m_s  speed_km_h  cantidad
0       1.39       5.004       237
1       2.78      10.008      7975
2       5.56      20.016      7405
3       8.33      29.988      7854
4      11.11      39.996       336
5      13.89      50.004      4692
6      16.67      60.012       244
7      22.22      79.992       624
8      27.78     100.008      2645


In [11]:
print("\n--- CARRILES POR EDGE ---")

print(
    edges_df["num_lanes"]
    .value_counts()
    .sort_index()
)


--- CARRILES POR EDGE ---
num_lanes
1     25174
2      2022
3       754
4        89
5        19
6         9
7         1
10        2
Name: count, dtype: int64


In [12]:
lanes_pct = (
    edges_df["num_lanes"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

print(lanes_pct)

num_lanes
1     89.682936
2      7.203420
3      2.686142
4      0.317064
5      0.067688
6      0.032063
7      0.003563
10     0.007125
Name: proportion, dtype: float64


In [13]:
print("\n--- TIPOS DE VÍA SUMO ---")

print(
    edges_df["type"]
    .value_counts(dropna=False)
)


--- TIPOS DE VÍA SUMO ---
type
highway.residential       8727
highway.footway           7367
highway.service           5870
highway.primary           1326
highway.cycleway          1187
highway.secondary          985
highway.tertiary           968
highway.pedestrian         368
highway.path               276
highway.steps              237
highway.trunk              235
highway.primary_link       164
highway.track              162
highway.trunk_link          95
highway.unclassified        77
highway.secondary_link      18
highway.bridleway            4
highway.tertiary_link        3
highway.service|psv          1
Name: count, dtype: int64


In [14]:
print("\n--- ALLOW ---")

print(
    lanes_df["allow"]
    .value_counts(dropna=False)
)

print("\n--- DISALLOW ---")

print(
    lanes_df["disallow"]
    .value_counts(dropna=False)
)


--- ALLOW ---
allow
NaN                                    16376
pedestrian                              7986
pedestrian delivery bicycle             5810
pedestrian bicycle                      1062
bicycle                                  427
pedestrian motorcycle moped bicycle      158
bus bicycle                              155
emergency authority bus bicycle           24
delivery bicycle                          14
Name: count, dtype: int64

--- DISALLOW ---
disallow
NaN                                                                                                                                  15636
tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone                       14711
pedestrian tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone              828
pedestrian bicycle tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft

In [15]:
def contiene_modo(valor, modo):
    if pd.isna(valor):
        return False

    return modo in str(valor).split()


lanes_df["permite_auto"] = (
    lanes_df["allow"].apply(lambda x: contiene_modo(x, "passenger"))
)

lanes_df["permite_bici"] = (
    lanes_df["allow"].apply(lambda x: contiene_modo(x, "bicycle"))
)

lanes_df["permite_peaton"] = (
    lanes_df["allow"].apply(lambda x: contiene_modo(x, "pedestrian"))
)

print("\n--- ACCESIBILIDAD ---")

print(
    lanes_df[
        ["permite_auto", "permite_bici", "permite_peaton"]
    ].sum()
)


--- ACCESIBILIDAD ---
permite_auto          0
permite_bici       7650
permite_peaton    15016
dtype: int64


In [16]:
print(
    lanes_df[
        ["permite_auto", "permite_bici", "permite_peaton"]
    ].mean() * 100
)

permite_auto       0.000000
permite_bici      23.897289
permite_peaton    46.907410
dtype: float64


In [17]:
accesibilidad = (
    lanes_df
    .groupby("type")
    .agg(
        carriles=("lane_id", "count"),
        velocidad_promedio=("speed", lambda x: x.astype(float).mean()),
        permite_auto=("permite_auto", "mean"),
        permite_bici=("permite_bici", "mean"),
        permite_peaton=("permite_peaton", "mean")
    )
)

accesibilidad["permite_auto"] *= 100
accesibilidad["permite_bici"] *= 100
accesibilidad["permite_peaton"] *= 100

accesibilidad

,carriles,velocidad_promedio,permite_auto,permite_bici,permite_peaton
type,,,,,
highway.bridleway,4,2.780000,0.0,0.000000,100.000000
highway.cycleway,1187,5.560000,0.0,100.000000,66.217355
highway.footway,7385,2.785261,0.0,0.000000,99.810427
highway.path,276,5.560000,0.0,100.000000,100.000000
highway.pedestrian,374,2.780000,0.0,0.000000,100.000000
highway.primary,3205,19.268577,0.0,3.400936,0.000000
highway.primary_link,251,19.320319,0.0,0.000000,0.000000
highway.residential,9220,10.833103,0.0,0.151844,0.000000
highway.secondary,1762,15.710289,0.0,0.567537,0.000000


In [18]:
print("\n--- ELEMENTOS RELACIONADOS CON ACERAS ---")

for elem in root.findall("edge"):
    edge_id = elem.get("id")

    for lane in elem.findall("lane"):
        allow = lane.get("allow", "")
        disallow = lane.get("disallow", "")

        if "pedestrian" in str(allow) or "pedestrian" in str(disallow):
            pass


--- ELEMENTOS RELACIONADOS CON ACERAS ---


In [19]:
print("\n--- NODOS CON CROSSING ---")

crossings = []

for junction in root.findall("junction"):

    for crossing in junction.findall("crossing"):

        crossings.append({
            "junction": junction.get("id"),
            "edges": crossing.get("edges"),
            "priority": crossing.get("priority"),
            "width": crossing.get("width"),
            "speed": crossing.get("speed")
        })

crossings_df = pd.DataFrame(crossings)

print(f"Cruces peatonales: {len(crossings_df):,}")

if len(crossings_df) > 0:
    display(crossings_df.head())


--- NODOS CON CROSSING ---
Cruces peatonales: 0


# Conversión de red OSM a SUMO y preparación multimodal

**Responsable:** Santiago Chitiva  
**Fase:** 2 - Preparación de datos (ETL)  
**Semana:** 3 
**Prerrequisito:** F1.3 - Red vial OSM validada (cerrado)

## Objetivo

Convertir la red vial de OpenStreetMap (OSM) correspondiente a la localidad de Usaquén al formato nativo de SUMO (`.net.xml`), ampliando el alcance de la red para considerar vehículos, bicicletas y peatones.

## Actividades realizadas

### 1. Revisión de la delimitación espacial

Se utilizó el límite oficial de la localidad de Usaquén empleado en F1.3, obtenido desde `loca.json` y filtrado mediante el atributo `LocNombre = 'USAQUEN'`.

El polígono fue transformado al sistema de coordenadas WGS84 (EPSG:4326) y se validó su geometría antes de utilizarlo para la consulta de OpenStreetMap.

### 2. Prueba exploratoria de la red con OSMnx

Inicialmente se utilizó OSMnx para consultar la red mediante `network_type="all"` y se comparó este resultado con un `custom_filter` que incluía categorías vehiculares y no motorizadas.

Se verificó que `network_type="all"` incorpora infraestructura relevante para el alcance multimodal, incluyendo:

- `footway`
- `cycleway`
- `path`
- `pedestrian`
- `steps`
- `corridor`
- `track`

además de las categorías de vías vehiculares convencionales.

### 3. Revisión de las categorías `highway`

Durante la comparación se identificó que OSMnx puede representar determinadas aristas con múltiples valores de `highway`, por ejemplo:

- `[residential, footway]`
- `[footway, steps]`
- `[residential, service]`
- `[footway, corridor]`

Se comprobó que estas combinaciones están relacionadas con la simplificación del grafo realizada por OSMnx. Al utilizar `simplify=False`, se conservaron los segmentos individuales y dejaron de aparecer las combinaciones de categorías como representación de una misma arista.

Sin embargo, esta configuración produjo una red considerablemente más fragmentada, por lo que no se utilizó como mecanismo definitivo para generar la red de SUMO.

### 4. Prueba inicial de conversión mediante OSMnx

Se realizó una primera prueba exportando el grafo de OSMnx mediante `ox.save_graph_xml()` y posteriormente convirtiéndolo a SUMO mediante `netconvert`.

La inspección del resultado en NetEdit mostró una representación diferente a la observada directamente en OSMnx, con numerosos segmentos que visualmente parecían desconectados.

Se verificó posteriormente la conectividad del grafo en OSMnx y se encontró que todos los nodos pertenecían a un único componente conectado, por lo que el problema visual no correspondía a una desconexión real del grafo.

### 5. Descarga directa de OSM mediante Overpass

Para evitar reconstruir el archivo OSM a partir del grafo de OSMnx, se utilizó el mismo polígono oficial de Usaquén para realizar una consulta directa a Overpass.

El flujo definitivo adoptado fue:

    Límite oficial de Usaquén
            ↓
    Polígono WGS84
            ↓
    Consulta Overpass
            ↓
    usaquen.osm.xml
            ↓
    netconvert
            ↓
    usaquen.net.xml

De esta manera, la conversión a SUMO utiliza directamente los datos OSM descargados para el área de estudio.

### 6. Generación de la red SUMO

Se generó una primera versión de la red mediante:

    netconvert --osm-files data/processed/usaquen.osm.xml `
      --output-file data/processed/usaquen.net.xml `
      --geometry.remove `
      --roundabouts.guess `
      --ramps.guess `
      --junctions.join `
      --tls.guess-signals `
      --tls.discard-simple

La red resultante fue inspeccionada mediante NetEdit y se verificó visualmente que presenta una estructura conectada y coherente con la red vial de la localidad.

### 7. Consideración de peatones y bicicletas

Debido a que el alcance de la simulación incluye vehículos, bicicletas y peatones, se mantuvieron categorías de infraestructura no motorizada presentes en OSM, entre ellas:

- `footway`
- `cycleway`
- `path`
- `pedestrian`
- `steps`
- `track`
- `corridor`

junto con las categorías vehiculares:

- `trunk`
- `primary`
- `secondary`
- `tertiary`
- `residential`
- `service`
- `unclassified`

Por lo tanto, la extracción ya no se limita exclusivamente a infraestructura destinada al tráfico vehicular motorizado.

## Resultados

Se generaron los siguientes archivos:

    data/processed/usaquen.osm.xml
    data/processed/usaquen.net.xml

El archivo `.net.xml` constituye la primera versión de prueba de la red de simulación SUMO para Usaquén.

La red obtenida conserva infraestructura relevante para el modelamiento multimodal de:

- Vehículos
- Bicicletas
- Peatones

## Hallazgos

- `network_type="all"` resulta más apropiado que un filtro exclusivamente vehicular debido al alcance multimodal definido para la simulación.
- Las categorías `footway`, `cycleway`, `path`, `pedestrian` y `steps` deben ser consideradas para representar adecuadamente los desplazamientos peatonales y ciclistas.
- La simplificación de OSMnx puede generar aristas con múltiples valores de `highway`; este comportamiento se evitó para el flujo definitivo al utilizar directamente los datos OSM descargados mediante Overpass.
- La red obtenida mediante OSMnx fue verificada como conectada, por lo que los segmentos visualmente aislados observados inicialmente no correspondían a componentes desconectados del grafo.
- La utilización directa del archivo OSM descargado mediante Overpass permitió obtener una conversión más adecuada para su posterior utilización en SUMO.
- Se generó y verificó visualmente una primera red `.net.xml` mediante NetEdit.

## Pendientes

1. Analizar los atributos `maxspeed` y `lanes` de la red.
2. Identificar el porcentaje de tramos que no cuentan con estos atributos.
3. Definir y documentar valores por defecto para los atributos faltantes.
4. Revisar las capacidades de las diferentes categorías de vías.
5. Configurar la accesibilidad específica para vehículos, bicicletas y peatones.
6. Revisar la generación de aceras y cruces peatonales.
7. Validar la red multimodal resultante en NetEdit y posteriormente en SUMO.

## Estado

**Primera red `.net.xml` generada y validada visualmente en NetEdit.**